In [ ]:
from recipeprep.config import get_config
config = get_config()
config.ensure_output_directories()

In [ ]:
import openai
import json
import random
import importlib
import numpy as np
from datetime import datetime
import glob

In [ ]:
from openai import OpenAI

In [ ]:
# Set OPENAI_API_KEY in your environment before running this notebook.

In [ ]:
importlib.reload(recipe_dataset_helper)

In [ ]:
from recipeprep.data import dataset_sampling as recipe_dataset_helper
from recipeprep.data import nutrient_client as ingre_nut_map_helper
from recipeprep.retrieval import index as sim_search_helper

In [ ]:
sample_size = 250 #testing recipe numbers
recipe_filename = config.raw_recipes_dir / 'recipes_raw_processed.json'
recipe_dataset_small_name = config.datasets_dir / f'recipe_dataset_init_{sample_size}.json'
long_recipe_percnt= 0.2
processed_init_filename = config.datasets_dir / f"processed_recipes_init_{sample_size}.json"

In [ ]:
#Init the client

temp = 0.7
topp = 1

In [ ]:
embd_name_1 = config.embeddings_dir / config.retrieval.description_embeddings_filename

In [ ]:
ingre_sim_search_client = OpenAI()  # Reads OPENAI_API_KEY from the environment

In [ ]:
recipe_data_process_client = OpenAI()  # Reads OPENAI_API_KEY from the environment

## Get a smaller recipe dataset

The original RecipeBox dataset contains around 100k datasets, we don't need that much for now

call get_long_short_recipe_dataset() to get a smaller dataset for testing

In [ ]:
recipe_dataset_helper.get_long_short_recipe_dataset(recipe_filename,sample_size,recipe_dataset_small_name,long_recipe_percnt=long_recipe_percnt)

## Data Processing

In [ ]:
from recipeprep.data.recipe_processing import (
    get_API_response,
    get_processed_recipe_dataset,
    process_API_res_get_processed_recipe,
)

### Recipe Data Initial Processing

Parse paragraph style instructions in structured labels

In [ ]:
# Recipe response parsing is implemented in recipeprep.data.recipe_processing.

In [ ]:
# Batch processing is implemented in recipeprep.data.recipe_processing.

In [ ]:
#Get testing file
print(recipe_dataset_small_name)
with open(recipe_dataset_small_name, "r") as f:
    recipe_dataset_TBP = json.load(f)

In [ ]:
from recipeprep.generation.prompts import RECIPE_PROCESS_PROMPT

# Keep the original notebook variable name for the cells below.
recipe_process_prompt = RECIPE_PROCESS_PROMPT

In [ ]:
test_recipes = dict(recipe_dataset_TBP)
batch_counter = get_processed_recipe_dataset(
    recipe_data_process_client,
    temp,
    topp,
    test_recipes,
    batch_size=50,
    prompt_template=recipe_process_prompt,
)

### Ingredient-Nutrient Mapping


Use https://produits-sante.canada.ca/api/documentation/cnf-documentation-en.html

Step1: Use the food-code dataset and configured embedding model to perform similarity matching and retrieve the closest food code and description.

Step2: Use the food_code to get nutrient amount(s) 

Step 3: Save the mapping for the ingredient in the ingre_nutrition_map folder


#### Get Food code description embedding

This Section DON"T NEED TO BE RUN EVERYTIME ! Otherwise the entire food code dataset will be processed -> cost 

Only run this section when have a new food_code dataset

In [ ]:
food_descriptions,food_codes = sim_search_helper.get_normalized_foodCode_dataset()
print(f'Number of food codes: {len(food_descriptions)}')

In [ ]:
from recipeprep.retrieval.embeddings import (
    batch_generate_embeddings,
    generate_embeddings,
)

In [ ]:
# Batch embedding is implemented in recipeprep.retrieval.embeddings.

In [ ]:
#!!!!!
#!!!! Only run this section when have a new food_code dataset
#!!!!!
food_embeddings = batch_generate_embeddings(ingre_sim_search_client, food_descriptions, batch_size=400)
food_embeddings = np.array(food_embeddings, dtype="float32")

np.save(embd_name_1, food_embeddings)  # Save embeddings
print(f"Embeddings saved to {embd_name_1}")


In [ ]:
food_embeddings = np.load(embd_name_1)
index = sim_search_helper.create_FAISS_Index(food_embeddings)

#### Start Similarity search

In [ ]:
food_descriptions_ori,_ = sim_search_helper.get_regular_foodCode_dataset()

In [ ]:
from recipeprep.retrieval.ingredient_matcher import (
    IngredientMatcher,
    find_closest_food_code,
    get_food_code_for_ingredients,
)

ingredient_matcher = IngredientMatcher(
    client=ingre_sim_search_client,
    index=index,
    food_descriptions=food_descriptions,
    food_codes=food_codes,
    food_descriptions_ori=food_descriptions_ori,
)

##### Run Full list

In [ ]:
total_item_num = len(recipe_dataset_TBP)

#Get ingredients from all batch files
folder_path = config.processed_recipes_dir
total_ingre_list = []
for file_path in folder_path.glob("processed_recipes_init*.json"):
    print(f"Processing file: {file_path}")
    batch_ingre_list = recipe_dataset_helper.get_ingre_list_from_dataset(file_path)
    print(len(batch_ingre_list))
    total_ingre_list.extend(batch_ingre_list)
    
total_ingre_list.sort()
all_ingredients_list = set(total_ingre_list)

In [ ]:
all_ingredients_list = list(set(all_ingredients_list))
all_ingredients_list.sort()
print(len(all_ingredients_list))

In [ ]:
# Ingredient matching is implemented in recipeprep.retrieval.ingredient_matcher.

In [ ]:
ingre_food_code_result = get_food_code_for_ingredients(
    all_ingredients_list,
    ingredient_matcher,
)

##### Testing


In [ ]:
test_food = 'tomato'
matched_food_code, matched_food_description, similarity = ingredient_matcher.find_closest_food_code(test_food)
print(f'===={matched_food_description}')

##### No Contextual Input

In [ ]:
# Print the results
# Case1: with NO contectual input
with (config.reports_dir / "ingredient_match.log").open("w", encoding="utf-8") as log_file:
    local_timestamp = datetime.now()
    print(f"----------------------Local Timestamp: {local_timestamp}--------------------------", file=log_file)
    for ingredient, details in ingre_food_code_result.items():

        print(f"Ingredient: {ingredient}", file=log_file)
        print(f"Matched Description: [{details['description']}], Food Code: {details['food_code']}, Similarity: {details['similarity']:.2f}", file=log_file)
        print('\n', file=log_file)

### Create Ingredient-nutrient Mapping

In [ ]:
# ingre_food_code_result
nut_unit_map_name = ingre_nut_map_helper.get_unitMap_name()
ntri_unit_map = ingre_nut_map_helper.load_nut_id_map(nut_unit_map_name)

In [ ]:
from recipeprep.data.nutrient_mapping import get_all_ingredient_mapping

In [ ]:
all_mapping,untri_unit_map = get_all_ingredient_mapping(ingre_food_code_result,ntri_unit_map)

In [ ]:
#Save map
ingre_nut_map_helper.save_nut_map(untri_unit_map,all_mapping)
local_timestamp = datetime.now()
print("Local Timestamp:", local_timestamp)